In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from basin_volume import DependenceEstimator, DependenceVolumeConfig

# Load any CausalLM model, tokenizer, and dataset
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-14m")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token_id = 1  # pythia-specific
tokenizer.eos_token_id = 0  # pythia-specific
dataset = load_dataset("EleutherAI/lambada_openai", name="en", split="test", trust_remote_code=True)
dataset2 = load_dataset("EleutherAI/lambada_openai", name="de", split="test", trust_remote_code=True)

# Configure the estimator
cfg = DependenceVolumeConfig(model=model, 
                   tokenizer=tokenizer, 
                   dataset=dataset, 
                   dataset2=dataset2,
                   text_key="text",  # must match dataset field
                   text_key2="text",  # must match dataset2 field
                   n_samples=10,  # number of MC samples
                   cutoff=1e-2,  # KL-divergence cutoff (nats)
                   max_seq_len=2048,  # max sequence length for tokenizer or chunk_and_tokenize
                   val_size=10,  # number of dataset sequences to use. default (None) uses all.
                   cache_mode=None,  # see below
                   chunking=False,  # whether to use chunk_and_tokenize
                   )
estimator = DependenceEstimator.from_config(cfg)

# Run the estimator
result = estimator.run()

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `GPTNeoXSdpaAttention` class is deprecated in favor of simply modifying the `config._attn_implementation`attribute of the `GPTNeoXAttention` class! It will be removed in v4.48


tokens.shape=torch.Size([5153, 224])


100%|██████████| 10/10 [00:10<00:00,  1.02s/it]


In [2]:
result

VolumeResult(estimates=tensor([-1.0854e+08, -1.0834e+08, -1.0739e+08, -1.0809e+08, -1.0748e+08,
        -1.0819e+08, -1.0884e+08, -1.0814e+08, -1.0874e+08, -1.0748e+08],
       device='cuda:0'), props=tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], device='cuda:0'), mults=tensor([0.5508, 0.5586, 0.5977, 0.5684, 0.5938, 0.5645, 0.5391, 0.5664, 0.5430,
        0.5938], device='cuda:0'), deltas=tensor([0.0101, 0.0099, 0.0101, 0.0100, 0.0099, 0.0100, 0.0101, 0.0100, 0.0099,
        0.0100], device='cuda:0'), logabsint=tensor([-15424106., -15225966., -14275080., -14982154., -14367332., -15079172.,
        -15726650., -15030575., -15625075., -14367329.], device='cuda:0'))